# Laboratorio de regresión - 5

|                |   |
:----------------|---|
| **Nombre**     |  Claudio Alejo Encarnación Martínez |
| **Fecha**      |  21 septiembre 2026 |
| **Expediente** |  750597 |

## Validación

Hemos estado usando `train_test_split` en nuestros modelos anteriores.

¿Por qué?

- **RESPUESTA**

Para analizar/detectar si los modelos que realizamos se desempeñan bien y no están sobreajustados. con este podemos poner a prueba un modelo con datos que no conoce (test) después de ser entrenados con una porción de los datos totales (train)

Si la muestra es un subset de la población y queremos generalizar sobre la población, ¿no sería mejor utilizar todos los datos al entrenar un modelo?

- **RESPUESTA**

Suena mejor, pero de esta forma nos quedamos sin datos "nuevos" con los cuales poner a prueba el modelo, si usáramos todos los datos para entrenarlo, lal momento de ponerlo a prueba lo estaríamos haciendo con datos que el modelo ya conoce, por lo que su desempeño podría ser muy bueno debido a que "memorizó" los datos, y no tanto porqu etenga una buena capacidad predictiva

El propósito de volver a muestrear dentro de nuestro dataset es tener una idea de qué tan buena podría ser la generalización de nuestro modelo. Imagina un dataset ya separado en dos mitades. Utilizas la primera mitad para entrenar el modelo y pruebas en la segunda mitad; la segunda mitad eran datos invisibles para el modelo al momento de entrenar. Esto nos lleva a tres escenario típicos:

1. Si el modelo hace buenas predicciones en la segunda mitad, significa que la primera mitad era "suficiente" para generalizar.
2. Si el modelo no hace buenas predicciones en la segunda mitad, pero sí en la primera mitad, podría ser que había información importante en la segunda mitad que debió haber sido tomada en cuenta al entrenar, o un problema de overfitting.
3. Si el modelo no hace buenas predicciones en la segunda mitad, y tampoco en la primera mitad, se tendrían que revisar los factores y/o el modelo seleccionado.

El caso ideal sería el 1, pero por estadística los errores y varianzas tienen como entrada el número de muestas, por lo que tenemos menos seguridad de nuestros resutados al usar menos muestras. Si vemos que el modelo generaliza bien podemos unir de nuevo el dataset y entrenar sobre el dataset completo.

En el caso 2 está el problema de que no podemos saber qué información es necesaria para el entrenamiento apropiado del modelo; esto nos lleva a pensar que debemos usar el dataset completo para entrenar, pero esto nos lleva al mismo problema de no saber si el modelo puede generalizar.

El problema sólo incrementa si se tienen hiperparámetros en el modelo (e.g. $\lambda$ en regularización).

## Leave-One-Out Cross Validation

Este método de validación es una colección de $n$ `train-test-split`. Teniendo un dataset de $n$ muestras, la lógica es:
1. Saca una muestra del dataset.
2. Entrena tu modelo con las $n-1$ muestras.
3. Evalúa tu modelo en la muestra que quedó fuera con el métrico que más se ajuste a la aplicación.
4. Regresa la muestra al dataset.
5. Repite 1-4 con muestras diferentes hasta haber hecho el procedimiento $n$ veces para $n$ muestras.
6. Calcula la media y desviación estándar de los métricos guardados.

Con los resultados del proceso de validación podemos saber qué tan bueno podría ser el modelo seleccionado con los datos (con/sin transformaciones).

### Ejercicio 1

Utiliza el dataset `Motor Trend Car Road Tests`. Elimina la columna `model` y entrena 32 modelos diferentes utilizando Leave-One-Out Cross Validation con target `mpg`. Utiliza MSE como métrico.

- **RESPUESTA**

In [12]:
import pandas as pd
import openpyxl
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np
from sklearn.linear_model import Lasso

In [16]:
df = pd.read_excel("/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/Motor Trend Car Road Tests.xlsx")
df.head(5)


,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [17]:
df = df.drop(columns=['model'])
df.head()

,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [18]:
X = df.drop(columns="mpg")
y = df["mpg"]

loo = LeaveOneOut()
predicciones = []
valores_reales = []

for train_idx, test_idx in loo.split(X):
    modelo = LinearRegression()
    modelo.fit(X.iloc[train_idx], y.iloc[train_idx])

    predicciones.append(modelo.predict(X.iloc[test_idx])[0])
    valores_reales.append(y.iloc[test_idx].values[0])

mse = mean_squared_error(valores_reales, predicciones)

print(f"Modelos entrenados: {loo.get_n_splits(X)}")
print(f"MSE promedio de LOOCV: {mse:.4f}")

df.head()

Modelos entrenados: 32
MSE promedio de LOOCV: 12.1816


,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


In [ ]:


X_mpg = df.drop(columns="mpg")
y_mpg = df["mpg"]

predicciones_lasso = []
valores_reales_lasso = []

for train_idx, test_idx in loo.split(X_mpg):
    modelo = Lasso(alpha=0.01, max_iter=10000)
    modelo.fit(X_mpg.iloc[train_idx], y_mpg.iloc[train_idx])

    predicciones_lasso.append(modelo.predict(X_mpg.iloc[test_idx])[0])
    valores_reales_lasso.append(y_mpg.iloc[test_idx].values[0])

mse_lasso = mean_squared_error(valores_reales_lasso, predicciones_lasso)

mse_lasso_folds = (
    np.array(valores_reales_lasso) - np.array(predicciones_lasso)
) ** 2

print(f"Modelos entrenados: {loo.get_n_splits(X_mpg)}")
print(f"MSE promedio de LOOCV con Lasso: {mse_lasso:.4f}")
print(f"Desviación estándar del MSE: {mse_lasso_folds.std():.4f}")

Modelos entrenados: 32
MSE promedio de LOOCV con Lasso: 11.3578


NameError: name 'mse_lasso_folds' is not defined

Interpreta.

In [7]:
print("El RMSE es de:", np.sqrt(mse))

El RMSE es de: 3.4902088772596334


- **RESPUESTA**

El modelo obtuvo un MSE de 12.18 y un RMSE de aproximadamente 3.49 mpg mediante LOOCV.
Esto significa que las predicciones se alejan típicamente 3.49 mpg del valor real, lo cual no es taaan malo ya que los valores de mpg van de 10 a 34 aproximadamente. Pero para determinar mejor si es un buen resultado, se debe comparar este RMSE con un modelo baseline o con otros modelos.

## K-Folds Cross-Validation

El dataset `Motor Trend Car Road Tests` sólo tiene 32 muestras, y utilizar un modelo sencillo de regresión múltiple hace que usar LOOCV sea muy rápido. El dataset `California Housing` tiene $20640$ muestras para $9$ columnas, entonces realizar un ajuste sobre una transformación o sobre el modelo y luego calcular el impacto esperado podría tomar más tiempo.

La solución propuesta es dividir el dataset en *k* folds (partes iguales), ajustar en *k-1* folds y probar en el restante.

### Ejercicio 2
Utiliza el dataset `California Housing` y haz K-folds Cross Validation con 10 folds. Utiliza el MSE como métrico.

In [8]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
print("Dataset Shape:", housing.data.shape, housing.target.shape)
print("Dataset Features:", housing.feature_names)
print("Dataset Target:", housing.target_names)
X = housing.data
y = housing.target

Dataset Shape: (20640, 8) (20640,)
Dataset Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Dataset Target: ['MedHouseVal']


- **RESPUESTA**

In [10]:
from sklearn.model_selection import KFold, cross_val_score

In [11]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

modelo_kfold = LinearRegression()
mse_folds = -cross_val_score(
    modelo_kfold,
    X,
    y,
    cv=kfold,
    scoring="neg_mean_squared_error"
)

print("MSE por fold:", mse_folds)
print(f"MSE promedio: {mse_folds.mean():.4f}")
print(f"Desviación estándar del MSE: {mse_folds.std():.4f}")

MSE por fold: [0.55900192 0.5531233  0.48116128 0.57420367 0.48897584 0.52838818
 0.55117824 0.47483961 0.56436922 0.55002537]
MSE promedio: 0.5325
Desviación estándar del MSE: 0.0352


Interpreta.

- **RESPUESTA**

Como los MSE de los folds que se hicieron son similares, se puede asumir que el modelo no depende de alguna partición particular de los datos. Si esto no fuera así, se observaría algún MSE mucho más disparado. 

## Referencia

James, G., Witten, D., Hastie, T., Tibshirani, R.,, Taylor, J. (2023). An Introduction to Statistical Learning with Applications in Python. Cham: Springer. ISBN: 978-3-031-38746-3